# OpenWEC — Driver Career Analysis

This notebook demonstrates how to analyze a driver's career across multiple series and seasons using the OpenWEC SDK and REST API.

**What we'll cover:**
- Loading a driver profile and career stats
- Analyzing race history across WEC, ELMS, and IMSA
- Comparing consistency across different circuits and conditions
- Visualizing career progression

**Requirements:**
```bash
pip install openwec[plotting] requests
```

## 1. Setup

In [ ]:
import openwec
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import httpx
import os

# Configure SDK
# Request a free key at https://openwec.com/api-keys
API_KEY  = os.environ.get("API_KEYS", "")
BASE_URL = "https://api.openwec.com/api/v1"

openwec.configure(api_key=API_KEY)

# Direct API client for endpoints not in the SDK
headers = {"X-API-Key": API_KEY} if API_KEY else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30)

print(f"openwec {openwec.__version__}")

## 2. Load driver profile

We'll use Filipe Albuquerque — one of the most experienced endurance drivers in the dataset, active across WEC and IMSA.

In [ ]:
# Driver ID 84 = Filipe Albuquerque
# Find driver IDs via the API: GET /series/WEC/seasons/2026/events/{id}/sessions/{id}/results
DRIVER_ID = 84

r = client.get(f"/drivers/{DRIVER_ID}")
profile = r.json()

print(f"Driver: {profile['first_name']} {profile['last_name']}")
print(f"Country: {profile['country']}")
print(f"Total races: {profile['total_races']}")
print(f"Series: {', '.join(profile['series'])}")
print(f"Classes: {', '.join(c for c in profile['classes'] if c)}")
print(f"First race: {profile['first_race']}")
print(f"Last race:  {profile['last_race']}")

## 3. Race history

In [ ]:
r = client.get(f"/drivers/{DRIVER_ID}/results", params={"limit": 100})
history = pd.DataFrame(r.json())

print(f"{len(history)} races found")
history.head(10)

In [ ]:
# Races by series
print("Races by series:")
print(history["series"].value_counts().to_string())
print()

# Podiums
podiums = history[history["position"].notna() & (history["position"] <= 3)]
print(f"Podiums: {len(podiums)}")
print(f"Wins:    {len(history[history['position'] == 1])}")

In [ ]:
# Results distribution
classified = history[
    history["position"].notna() &
    history["status"].str.contains("Classified", case=False, na=False)
].copy()

classified["position"] = classified["position"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Position distribution
classified["position"].hist(bins=20, ax=axes[0], color="#FFB000", edgecolor="#14181F")
axes[0].set_facecolor("#1C222C")
axes[0].set_title("Finishing Position Distribution", color="#ECEFF4")
axes[0].set_xlabel("Position", color="#8B95A7")
axes[0].set_ylabel("Races", color="#8B95A7")
axes[0].tick_params(colors="#8B95A7")

# Races per series
series_counts = history["series"].value_counts()
series_counts.plot.bar(ax=axes[1], color="#4895EF", edgecolor="#14181F")
axes[1].set_facecolor("#1C222C")
axes[1].set_title("Races by Series", color="#ECEFF4")
axes[1].set_xlabel("", color="#8B95A7")
axes[1].set_ylabel("Races", color="#8B95A7")
axes[1].tick_params(colors="#8B95A7", axis='x', rotation=0)

fig.patch.set_facecolor("#14181F")
plt.tight_layout()
plt.show()

## 4. Consistency analysis across sessions

Uses the protected `/drivers/{id}/consistency` endpoint to compare driving consistency across multiple race sessions.

In [ ]:
headers = {"X-API-Key": API_KEY}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30)
openwec.configure(api_key=API_KEY)
print("Client recriado")

In [ ]:
if not API_KEY:
    print("API key required for consistency data.")
    print("Request one at https://openwec.com/api-keys")
else:
    r = client.get(f"/drivers/{DRIVER_ID}/consistency", params={"limit": 50})
    consistency = pd.DataFrame(r.json())
    print(f"{len(consistency)} sessions with consistency data")
    consistency[["series", "season", "event", "car_class",
                 "avg_pace_s", "consistency_s", "green_flag_laps"]].head(10)
    


In [ ]:

if API_KEY and not consistency.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#14181F")
    ax.set_facecolor("#1C222C")

    for series, group in consistency.groupby("series"):
        ax.scatter(
            group["avg_pace_s"],
            group["consistency_s"],
            label=series,
            alpha=0.7,
            s=60
        )

    ax.set_xlabel("Average Pace (s)", color="#8B95A7")
    ax.set_ylabel("Consistency σ (s)", color="#8B95A7")
    ax.set_title(
        f"{profile['first_name']} {profile['last_name']} — Pace vs Consistency",
        color="#ECEFF4"
    )
    ax.tick_params(colors="#8B95A7")
    ax.legend(facecolor="#232A36", labelcolor="#ECEFF4")
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

    print(f"\nBest consistency: {consistency['consistency_s'].min():.3f}s")
    print(f"Avg consistency:  {consistency['consistency_s'].mean():.3f}s")

## 5. Deep dive — a specific race

Load lap data from a specific session where this driver competed.

In [ ]:
# Find a WEC race from the history
wec_races = history[history["series"] == "WEC"].head(5)
print("Recent WEC races:")
print(wec_races[["season", "event", "car_number", "car_class", "position"]].to_string(index=False))

In [ ]:
if API_KEY:
    # Load Le Mans 2026 — pick the car this driver was in
    le_mans = openwec.Session("WEC", 2026, "Le Mans", "Race")

    # Find which car Albuquerque drove
    results = le_mans.results()
    driver_car = results[results["drivers"].str.contains("Albuquerque", case=False, na=False)]

    if not driver_car.empty:
        car_number = driver_car.iloc[0]["car_number"]
        print(f"Albuquerque drove car #{car_number} at Le Mans 2026")
        print(f"Position: {driver_car.iloc[0]['position']}")
        print(f"Team: {driver_car.iloc[0]['team']}")

        # Load laps
        laps = le_mans.laps(car=str(car_number))
        fig = le_mans.plot_lap_evolution(car=str(car_number))
        fig.suptitle(f"Le Mans 2026 — Car #{car_number} Lap Evolution", y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print("Driver not found in Le Mans 2026 results — check driver ID or season")

## 6. Finding drivers by name

The API doesn't have a driver search endpoint yet, but you can browse results to find driver IDs.

In [ ]:
# Get all drivers from a specific race and find their IDs
session = openwec.Session("WEC", 2026, "Le Mans", "Race")
results = session.results()

# Show HYPERCAR drivers
hypercar = results[results["car_class"] == "HYPERCAR"][["car_number", "team", "drivers", "position"]]
print("HYPERCAR drivers at Le Mans 2026:")
print(hypercar.to_string(index=False))

---

## Next steps

- [API Documentation](https://api.openwec.com/docs) — browse all available endpoints
- [le_mans_2026.ipynb](le_mans_2026.ipynb) — full race analysis notebook
- [openwec.com](https://openwec.com) — live dashboard
- [Request API key](https://openwec.com/api-keys) — for consistency and analytics data